In [4]:
import pandas as pd
DataFrame = pd.read_csv("customer_shopping_behavior.csv")
#Tabla head
print(DataFrame.head())
#Tabla Datos Columna

   Customer ID  Age Gender Item Purchased  Category  Purchase Amount (USD)  \
0            1   55   Male         Blouse  Clothing                     53   
1            2   19   Male        Sweater  Clothing                     64   
2            3   50   Male          Jeans  Clothing                     73   
3            4   21   Male        Sandals  Footwear                     90   
4            5   45   Male         Blouse  Clothing                     49   

        Location Size      Color  Season  Review Rating Subscription Status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   Shipping Type Discount Applied Promo Code Used  Previ

In [5]:
print(DataFrame.info())
#Tabla  Etadistica
print(DataFrame.describe())
print(DataFrame.describe(include='all'))
#valores nulos
print(DataFrame.isnull().sum())

#LLenar los nulos en Review Rating
# Por categoria sacar una mediana de calificacion
DataFrame['Review Rating'] = DataFrame.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))
print(DataFrame.isnull().sum())

#Formatear Nombre de Columnas a minusculas
DataFrame.columns = DataFrame.columns.str.lower()
DataFrame.columns = DataFrame.columns.str.replace(' ' ,'_')
DataFrame = DataFrame.rename(columns={'purchase_amount_(usd)':'purchase_amount'})
print(DataFrame.columns)

#crear Nueva columnab age_group
labels = ['Young Adult', 'Adult','Middle-aged','Senior']
DataFrame['age_group'] = pd.qcut(DataFrame['age'], q=4, labels= labels)
print(DataFrame [['age','age_group']].head(10))

#Crar Column Dias de Frecuencia de Compra
frequency_mapping = {
    'Fortnightly' : 14,
    'Weekly':7,
    'Monthly':30,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Annually':365,
    'Every 3 Months': 90
}
DataFrame['purchase_frequency_days'] = DataFrame ['frequency_of_purchases'].map(frequency_mapping)
print(DataFrame[['purchase_frequency_days','frequency_of_purchases']].head(10))

# descuento == codigo promocional (valoores)
print(DataFrame[['discount_applied','promo_code_used']].head(10))
print((DataFrame['discount_applied'] == DataFrame['promo_code_used']).all)

#eliminar columan
DataFrame = DataFrame.drop('promo_code_used', axis=1)
print(DataFrame.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [6]:
pip install psycopg2-binary sqlalchemy


   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.8 MB 6.0 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.8 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 5.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [9]:
from sqlalchemy import create_engine

#Step: Coneecion a PostgreSQL
username = 'postgres'
password = '200207'
host = 'localhost'
port = '5432'
database = 'customer_behavior'

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

#Step: Cargar Datos DataFrame
table_name = 'customer'
DataFrame.to_sql(table_name,engine,if_exists='replace',index=False)
print(f"Data Successfully loaded into table'{table_name}' in database '{database}'")

Data Successfully loaded into table'customer' in database 'customer_behavior'
